# Dependencies

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import researchpy as rp
import statsmodels.api as sm
from sklearn import linear_model, datasets
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.pipeline import Pipeline
import seaborn as sns
from scipy import stats
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.datasets import make_regression
from sklearn.metrics import mean_squared_error
from sklearn.inspection import permutation_importance


In [ ]:
random_state = 7 # 6

# Read the data

In [ ]:
data_dir = "data_dir"
file_name = data_dir + "\\all_percentages_all_cohort.csv"

In [ ]:
df = pd.read_csv(file_name, index_col=0)
df['id'] = df.Dataset + "_" + df.donor_id
df = df.pivot_table(index=['donor_id','Dataset', 'Age', 'Ethnicity', 'Sex'], columns='Annotation', values='percent')
df= df.reset_index()

has_na_flag = df.isna().sum(axis=1) > 0
rows_with_na = has_na_flag[has_na_flag == True].index
df

In [ ]:
df.shape

# Visualize the data

In [ ]:
X = df.drop(["donor_id", "Age", "Dataset", "Annotation", "donor_id", "Ethnicity", "Sex"], axis=1, errors='ignore' ).astype(float)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled[np.isnan(X_scaled)] = 0

tsne = TSNE(random_state=random_state)
tsne_repr = tsne.fit_transform(X_scaled)

##  PCA

In [ ]:
pca = PCA(n_components=5, random_state=random_state)
pca_repr = pca.fit_transform(X_scaled)
new_df = pd.DataFrame(pca_repr)
new_df['Dataset'] = df.Dataset.values
new_df['Sex'] = df.Sex.values
new_df['Ethnicity'] = df.Ethnicity.values
sns.scatterplot(x=0, y=1, hue="Dataset", data=new_df)
sns.pairplot(new_df, hue="Dataset", plot_kws={"s": 5})
sns.pairplot(new_df, hue="Ethnicity", plot_kws={"s": 5})
sns.pairplot(new_df, hue="Sex", plot_kws={"s": 5})


## TSNE

In [ ]:
plt.scatter(tsne_repr[:, 0], tsne_repr[:, 1], alpha=0.5);
new_df = pd.DataFrame(tsne_repr)
new_df['Dataset'] = df.Dataset.values
new_df['Sex'] = df.Sex.values
new_df['Ethnicity'] = df.Ethnicity.values

fig, ax = plt.subplots()
sns.scatterplot(x=0, y=1, hue="Dataset", data=new_df, ax=ax)
fig, ax = plt.subplots()
sns.scatterplot(x=0, y=1, hue="Sex", data=new_df,ax=ax)
fig, ax = plt.subplots()
sns.scatterplot(x=0, y=1, hue="Ethnicity", data=new_df,ax=ax)

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(8, 6))
selected_column =  'CD8 Tmem KLRC2+'
sns.scatterplot(data=df, x='Age', y=selected_column, hue='Dataset', ax=axes)
sns.regplot(data=df[df.Dataset == 'OneK1K'], x='Age', y=selected_column, ax=axes, scatter=False, ci=0.1)
sns.regplot(data=df[df.Dataset == 'ABF300'], x='Age', y=selected_column, ax=axes, scatter=False, ci=0.1)
sns.regplot(data=df[df.Dataset == 'SoundLife'], x='Age', y=selected_column, ax=axes, scatter=False, ci=0.1)
sns.regplot(data=df[df.Dataset == 'UCSF'], x='Age', y=selected_column, ax=axes, scatter=False, ci=0.1)
sns.regplot(data=df[df.Dataset == 'AIDA'], x='Age', y=selected_column, ax=axes, scatter=False, ci=0.1)
sns.regplot(data=df[df.Dataset == 'SPAC'], x='Age', y=selected_column, ax=axes, scatter=False, ci=0.1)
sns.regplot(data=df[df.Dataset == 'KCL'], x='Age', y=selected_column, ax=axes, scatter=False, ci=0.1)
sns.regplot(data=df[df.Dataset == 'CIMA'], x='Age', y=selected_column, ax=axes, scatter=False, ci=0.1)



In [ ]:
np.unique(df.columns)

# Regression
## Util methods

In [ ]:

feature_columns =list(df.columns)[2:-1]
def plot_model_estimation(regr, y_test, y_pred, X_test, feature_names=feature_columns,
                          plot_important_features = False,
                          title="Regression estimation plot", additional_feature=None, point_size=5):
    # The coefficients
    fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(10, 6))
    fig.suptitle(title)
    values = np.vstack([y_test, y_pred])
    kernel = stats.gaussian_kde(values)(values)

    sns.scatterplot(x=y_test, y=y_pred, ax=axes[0, 0],    c=kernel,
    cmap="viridis")

    sns.kdeplot(x=y_test, y=y_pred, ax=axes[0, 0],levels=5,fill=True,alpha=0.6, cut=2)
    sns.lineplot(x=y_test, y=y_test, ax=axes[0, 0], color="red", label="true")
    sns.regplot(x=y_test, y=y_pred,  ax=axes[0, 0], scatter=False, color="blue", ci=None, robust=True)
    sns.lineplot(x=y_test, y=y_pred, ax=axes[0, 0], color="green", label="predicted")
    axes[0, 0].set(xlabel="true age", ylabel="predicted age")
    axes[0, 0].legend().set_visible(False)
    axes[0, 1].set_visible(False)
    axes[0, 1].legend().set_visible(False)
    sns.scatterplot(x=X_test[feature_names[0]], y=y_test, ax=axes[0, 2])
    sns.lineplot(x=X_test[feature_names[0]], y=y_test, ax=axes[0, 2], color="red", label="true")
    sns.lineplot(x=X_test[feature_names[0]], y=y_pred, ax=axes[0, 2], color="green", label="predicted")
    #sns.regplot(x=X_test[:,0], y=y_pred, ax=axes[0,1], scatter=False, color="green", ci=10)
    axes[0, 2].set(xlabel=feature_names[0], ylabel="true age")
    axes[0, 2].legend().set_visible(False)
    if plot_important_features:
        if (len(feature_names) > 1):
            sns.scatterplot(x=X_test[feature_names[1]], y=y_test, ax=axes[1, 0])
            sns.lineplot(x=X_test[feature_names[1]], y=y_test, ax=axes[1, 0], color="red", label="true")
            sns.lineplot(x=X_test[feature_names[1]], y=y_pred, ax=axes[1, 0], color="green", label="predicted")
            #sns.regplot(x=X_test[:,0], y=y_pred, ax=axes[0,1], scatter=False, color="green", ci=10)
            axes[1, 0].set(xlabel=feature_names[1], ylabel="true age")
            axes[1, 0].legend().set_visible(False)
        if (len(feature_names) > 2):
            sns.scatterplot(x=X_test[feature_names[2]], y=y_test, ax=axes[1, 1])
            sns.lineplot(x=X_test[feature_names[2]], y=y_test, ax=axes[1, 1], color="red", label="true")
            sns.lineplot(x=X_test[feature_names[2]], y=y_pred, ax=axes[1, 1], color="green", label="predicted")
            #sns.regplot(x=X_test[:,0], y=y_pred, ax=axes[0,1], scatter=False, color="green", ci=10)
            axes[1, 1].set(xlabel=feature_names[2], ylabel="true age")
            axes[1, 1].legend().set_visible(False)
        if (len(feature_names) > 3):
            sns.scatterplot(x=X_test[feature_names[3]], y=y_test, ax=axes[1, 2])
            sns.lineplot(x=X_test[feature_names[3]], y=y_test, ax=axes[1, 2], color="red", label="true")
            sns.lineplot(x=X_test[feature_names[3]], y=y_pred, ax=axes[1, 2], color="green", label="predicted")
            #sns.regplot(x=X_test[:,0], y=y_pred, ax=axes[0,1], scatter=False, color="green", ci=10)
            axes[1, 2].set(xlabel=feature_names[3], ylabel="true age")
            axes[1, 2].legend().set_visible(False)

    # The mean squared error
    print("MSE: %.2f" % mean_squared_error(y_test, y_pred))
    # MAE
    print("МАE: %.2f" % mean_absolute_error(y_test, y_pred))
    # RMSE
    print("RMSE: %.2f" % root_mean_squared_error(y_test, y_pred))
    # The coefficient of determination: 1 is perfect prediction
    print("R^2: %.2f" % r2_score(y_test, y_pred))

    handles, labels = plt.gca().get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center')
    plt.tight_layout()



    # if (additional_feature is not None):
    #     fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(10, 10))
    #     data_df = pd.DataFrame({"y_actual": y_test, "y_pred": y_pred, "dataset": additional_feature})
    #
    #     unique_datasets = np.unique(data_df.dataset)
    #     sns.kdeplot(data=data_df, x="y_actual", y="y_pred", hue="dataset", hue_order=unique_datasets, ax=axes,levels=5,fill=False,alpha=0.6, cut=2)
    #
    #     for dataset, current_color in zip(unique_datasets, sns.color_palette()):
    #         current_data = data_df[data_df.dataset == dataset]
    #         values = np.vstack([current_data.y_actual.values, current_data.y_pred.values])
    #         kernel = stats.gaussian_kde(values)(values)
    #         # sns.scatterplot(data=current_data, x="y_actual", y="y_pred", ax=axes, label=dataset, color=current_color, cmap="viridis", s=3)
    #
    #         sns.regplot(data=current_data, x="y_actual", y="y_pred",  ax=axes, scatter=False, color=current_color, label=dataset, ci=None, robust=True)
    #
    #         #sns.lmplot(data=current_data, x="y_actual", y="y_pred",  hue="dataset", scatter=True,  ci=None, robust=True)
    #     sns.set_style("ticks")

    sns.despine()

    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(10, 10))
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    sns.kdeplot(x=y_test, y=y_pred, ax=ax,levels=8,fill=False,alpha=0.6, cut=2)
    values = np.vstack([y_test, y_pred])
    kernel = stats.gaussian_kde(values)(values)
    sns.scatterplot(x=y_test, y=y_pred,  ax=ax, s=point_size,   c=-kernel, cmap=sns.color_palette("Spectral", as_cmap=True))
    xpoints = ypoints = ax.get_xlim()
    sns.lineplot(x=xpoints, y=ypoints, ax=ax, color="red",  linestyle='dashed')
    ax.set(xlabel="true age", ylabel="predicted age")
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)





class Objective(object):
    def __init__(self, data_X, data_Y):
        self.data_X = data_X
        self.data_Y = data_Y

    def __call__(self, trial):
        X_train, y_train = self.data_X, self.data_Y


        max_depth = trial.suggest_int("max_depth", 2, 5)
        n_estimators = trial.suggest_int("n_estimators", 1, 30)
        min_samples_split = trial.suggest_float("min_samples_split", 0.02, 0.2)
        max_features = trial.suggest_int("max_features", 4, 10)
        # Create the regressor
        regressor = RandomForestRegressor(
            max_depth=max_depth,
            n_estimators=n_estimators,
            min_samples_split=min_samples_split,
            max_features=max_features,
            random_state=42
        )
        # Evaluate the model using cross-validation
        score = cross_val_score(regressor, X_train, y_train, cv=3, scoring='neg_mean_absolute_error')
        error_value = -score.mean()
        return error_value


def search_hyperparams(data_df, dataset_name, ntrials=100):
    print ("*"*25)
    print(dataset_name)
    print ("*"*25)
    data_df = data_df.drop(columns=['id', 'index', 'donor_id', 'Dataset'], errors='ignore')
    target_df = data_df.copy()
    data_Y = target_df.age.values
    target_df = target_df.drop(columns=['age'])
    data_X = target_df
    objective = Objective(data_X, data_Y)
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=ntrials) # Run 50 trials
    # 4. Print the best hyperparameters and value
    print(f"Best trial value (MAE): {study.best_trial.value}")
    print(f"Best hyperparameters: {study.best_trial.params}")
    return study.best_trial.params


from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import RidgeCV, Ridge



## Get regressions and feature importance for separate datasets

In [ ]:
features_to_remove = ['donor_id', 'Dataset', 'Ethnicity', 'Sex']
df.columns

In [ ]:
class YoungRegressorSeparate(RandomForestRegressor):
    def fit(self, X, y, sample_weight=None):
        weights = [1 if (item <= 50) else 1e-10 for item in y]
        #print("sum of weights:", sum(weights))
        super().fit(X, y, sample_weight=weights)
        return self

class OldRegressorSeparate(RandomForestRegressor):
    def fit(self, X, y, sample_weight=None):
        weights = [1 if (item > 50) else 1e-10 for item in y]
        #print("sum of weights:", sum(weights))
        super().fit(X, y, sample_weight=weights)
        return self

def train_separate_datasets_regressions(df, features_to_remove):
    per_dataset_importances_test = dict()
    per_dataset_importances_sd_test = dict()
    per_dataset_importances_mae = dict()
    per_dataset_importances_sd_mae = dict()
    per_dataset_importances_train = dict()
    per_dataset_importances_sd_train = dict()
    per_dataset_coefs = dict()
    per_dataset_data_X_train = dict()
    per_dataset_data_X_test = dict()
    per_dataset_data_Y_train = dict()
    per_dataset_data_Y_test = dict()
    per_dataset_model = dict()

    for cur_dataset in df.Dataset.unique():
        print("*"*12)
        print(cur_dataset)
        print("*"*12)
        data_df = df
        data_df = data_df.fillna(0)
        target_df = data_df.copy()
        scaler = StandardScaler()
        power = PowerTransformer(method='box-cox')
        pipeline = Pipeline(steps=[('s', scaler)])


        data_df = data_df[data_df.Dataset == cur_dataset]
        # selected_params = search_hyperparams(data_df, cur_dataset)
        saved_data = data_df.copy()
        data_df = data_df.drop(columns=features_to_remove , errors='ignore')
        columns = data_df.columns
        target_df = data_df.copy()
        data_Y = target_df.Age.values
        target_df = target_df.drop(columns=['Age'])
        feature_names = target_df.columns
        # target_df = pd.DataFrame(pipeline.fit_transform(target_df), columns=feature_names)
        data_X = target_df
        X_train, X_test, y_train, y_test = train_test_split(data_X, data_Y, test_size=0.5, random_state=random_state)

        regr =  RandomForestRegressor(n_estimators=300, max_features=15,  criterion='squared_error', random_state=random_state)


        regr.fit(X_train, y_train)
        print("Score of single regression")
        print("Train")
        y_pred = regr.predict(X_train)
        print("MSE: %.2f" % mean_squared_error(y_train, y_pred))
        # MAE
        print("МАE: %.2f" % mean_absolute_error(y_train, y_pred))
        # RMSE
        print("RMSE: %.2f" % root_mean_squared_error(y_train, y_pred))
        # The coefficient of determination: 1 is perfect prediction
        print("R^2: %.2f" % r2_score(y_train, y_pred))
        y_pred = regr.predict(X_test)
        print("Test")
        print("MSE: %.2f" % mean_squared_error(y_test, y_pred))
        # MAE
        print("МАE: %.2f" % mean_absolute_error(y_test, y_pred))
        # RMSE
        print("RMSE: %.2f" % root_mean_squared_error(y_test, y_pred))
        # The coefficient of determination: 1 is perfect prediction
        print("R^2: %.2f" % r2_score(y_test, y_pred))

        ## Start over with new models
        regr_old = OldRegressorSeparate(n_estimators=300, max_features=15, criterion='squared_error', random_state = random_state)
        regr_young= YoungRegressorSeparate(n_estimators=300, max_features=15, criterion='squared_error', random_state=random_state)
        regr =  RandomForestRegressor(n_estimators=300, max_features=15,  criterion='squared_error', random_state= random_state)


        stacking_regressor = StackingRegressor(estimators = [
            ("general", regr), ("young", regr_young), ("old", regr_old)
        ], final_estimator=RidgeCV())

        stacking_regressor.fit(X_train, y_train)
        # Make predictions using the testing set
        #y_pred = regr.predict(X_train)
        #y_pred_young = regr.predict(X_train)
        #y_pred_old = regr.predict(X_train)
        y_pred = stacking_regressor.predict(X_train)
        #y_pred = (y_pred_young + y_pred_old) / 2
        #print("Coefficients: \n", np.round(regr.coef_, 2))
        regr = stacking_regressor.named_estimators_['general']

        #plot_model_estimation(regr, y_train, y_pred, X_train, feature_names=X_train.columns[list(reversed(np.argsort(regr.feature_importances_)))],  title=f"Data {cur_dataset} train")
        # Make predictions using the testing set
        y_pred = regr.predict(X_test)
        #print("Coefficients: \n", np.round(regr.coef_, 2))
        #plot_model_estimation(regr, y_test, y_pred, X_test, feature_names=X_train.columns[list(reversed(np.argsort(regr.feature_importances_)))], title=f"Data {cur_dataset} test")
        coef_dict = {column: np.round(importance,2) for column, importance in reversed(sorted(zip(X_train.columns,regr.feature_importances_), key=lambda x:x[1]))}
        coef_df = pd.DataFrame({"cluster":coef_dict.keys(), "importance": coef_dict.values()}).transpose()
        regr = stacking_regressor

        print("Result model")
        print("Train")
        y_pred = regr.predict(X_train)
        print("MSE: %.2f" % mean_squared_error(y_train, y_pred))
        # MAE
        print("МАE: %.2f" % mean_absolute_error(y_train, y_pred))
        # RMSE
        print("RMSE: %.2f" % root_mean_squared_error(y_train, y_pred))
        # The coefficient of determination: 1 is perfect prediction
        print("R^2: %.2f" % r2_score(y_train, y_pred))
        y_pred = regr.predict(X_test)
        print("Test")
        print("MSE: %.2f" % mean_squared_error(y_test, y_pred))
        # MAE
        print("МАE: %.2f" % mean_absolute_error(y_test, y_pred))
        # RMSE
        print("RMSE: %.2f" % root_mean_squared_error(y_test, y_pred))
        # The coefficient of determination: 1 is perfect prediction
        print("R^2: %.2f" % r2_score(y_test, y_pred))

        result = permutation_importance(
            regr, X_test, y_test, n_repeats=10, random_state=42, n_jobs=2
        )
        # print(f"Elapsed time to compute the importances: {elapsed_time:.3f} seconds")
        forest_importances_test = pd.Series(result.importances_mean, index=feature_names)
        forest_impotances_test_sd = pd.Series(forest_importances_test.std(), index=feature_names)

        forest_importances_test.sort_values(ascending=False, inplace=True)
        forest_impotances_test_sd = forest_impotances_test_sd.loc[forest_importances_test.index]

        per_dataset_importances_test[cur_dataset] = dict(forest_importances_test)
        per_dataset_importances_sd_test = forest_impotances_test_sd

        result = permutation_importance(
            regr, X_test, y_test, n_repeats=10, random_state=random_state, n_jobs=2, scoring='neg_mean_absolute_error'
        )
        # print(f"Elapsed time to compute the importances: {elapsed_time:.3f} seconds")
        forest_importances_test_mae = pd.Series(result.importances_mean, index=feature_names)
        forest_importances_test_mae.sort_values(ascending=False, inplace=True)
        forest_impotances_sd_test_mae = pd.Series(forest_importances_test_mae.std(), index=feature_names)

        per_dataset_importances_mae[cur_dataset] = dict(forest_importances_test_mae)
        per_dataset_importances_sd_mae[cur_dataset] = dict(forest_impotances_sd_test_mae)


        # result = permutation_importance(
        #     regr, X_train, y_train, n_repeats=10, random_state=random_state, n_jobs=2
        # )
        # print(f"Elapsed time to compute the importances: {elapsed_time:.3f} seconds")
        forest_importances_train = None
        forest_impotances_sd_train = None
        # forest_importances_train = pd.Series(result.importances_mean, index=feature_names)
        # forest_impotances_sd_train = pd.Series(forest_importances_train.std(), index=feature_names)
        #
        # forest_importances_train.sort_values(ascending=False, inplace=True)
        # forest_impotances_sd_train = forest_impotances_sd_train.loc[forest_importances_train.index]
        #
       # per_dataset_importances_train[cur_dataset] = dict(forest_importances_train)
        #per_dataset_importances_sd_train[cur_dataset] = dict(forest_impotances_sd_train)
        #per_dataset_coefs[cur_dataset] = dict(coef_dict)

        per_dataset_data_X_train = X_train
        per_dataset_data_y_train = y_train
        per_dataset_data_X_test = X_test
        per_dataset_data_y_test = y_test

    return (
        per_dataset_importances_test, #0
        per_dataset_importances_mae, #1
        per_dataset_importances_train, #2
        per_dataset_coefs, #3
        per_dataset_data_X_train, #4
        per_dataset_data_Y_train, #5
        per_dataset_data_X_test, #6
        per_dataset_data_Y_test, #7
        per_dataset_model, #8
        per_dataset_importances_sd_test, #9
        per_dataset_importances_sd_mae, #10
        per_dataset_importances_sd_train #11
    )



In [ ]:
result = train_separate_datasets_regressions(df, features_to_remove)
per_dataset_importances_test = result[0]
per_dataset_importances_mae = result[1]
per_dataset_importances_train = result[2]
per_dataset_coefs = result[3]
per_dataset_importances_test_sd = result[9]
per_dataset_importances_sd_mae = result[10]
per_dataset_importances_sd_train = result[11]


## Also combined a run

In [ ]:
class YoungRegressorCombined(RandomForestRegressor):
    def fit(self, X, y, sample_weight=None):
        weights = [1 if (item <= 50) else 0 for item in y]
        #print("sum of weights:", sum(weights))
        super().fit(X, y, sample_weight=weights)
        return self


class OldRegressorCombined(RandomForestRegressor):
    def fit(self, X, y, sample_weight=None):
        weights = [1 if (item > 50) else 0 for item in y]
       # print("sum of weights:", sum(weights))
        super().fit(X, y, sample_weight=weights)
        return self

def combined_regression_run(df, features_to_remove):
    data_df = df
    data_df = data_df.fillna(0)
    target_df = data_df.copy()
    scaler = StandardScaler()
    # power = PowerTransformer(method='box-cox')
    # pipeline = Pipeline(steps=[('s', scaler)])
    # selected_params = search_hyperparams(data_df, cur_dataset)
    saved_data = data_df.copy()
    data_df = data_df.drop(columns=features_to_remove , errors='ignore')
    target_df = data_df.copy()
    data_Y = target_df.Age.values
    target_df = target_df.drop(columns=['Age'])
    feature_names = target_df.columns
    # target_df = pd.DataFrame(pipeline.fit_transform(target_df), columns=feature_names)
    data_X = target_df
    X_train, X_test, y_train, y_test = train_test_split(data_X, data_Y, test_size=0.5, random_state=random_state)
    regr =  RandomForestRegressor(n_estimators=300, max_features=15,  criterion='squared_error', random_state=random_state)
    # regr = RandomForestRegressor(max_depth = selected_params['max_depth'],
    #                              n_estimators=selected_params['n_estimators'],
    #                              max_features=selected_params['max_features'],
    #                              min_samples_split=selected_params['min_samples_split'])
    y_train_dataset = saved_data.loc[X_train.index].Dataset.values
    bins = np.arange(min(y_train), max(y_train), 3)
    weighted_hist, bin_edges = np.histogram(y_train, bins=bins)
    indices = np.digitize(y_train, bins)
    indices = np.clip(indices, 1, len(weighted_hist))
    sample_weights = weighted_hist[indices - 1]
    sample_weights = 1/ sample_weights

    total_samples = len(y_test)
    num_sources = len(np.unique(y_train_dataset))

    source_counts = saved_data.loc[X_train.index].Dataset.value_counts()
    source_weights = total_samples / (num_sources * source_counts)
    train_source_weights = np.array(saved_data.loc[X_train.index].Dataset.map(source_weights))

    #regr.fit(X_train, y_train, sample_weight=100* sample_weights*train_source_weights)
    regr.fit(X_train, y_train)
    y_pred = regr.predict(X_test)
    print("Score of single regression")
    print("Train")
    y_pred = regr.predict(X_train)
    print("MSE: %.2f" % mean_squared_error(y_train, y_pred))
    # MAE
    print("МАE: %.2f" % mean_absolute_error(y_train, y_pred))
    # RMSE
    print("RMSE: %.2f" % root_mean_squared_error(y_train, y_pred))
    # The coefficient of determination: 1 is perfect prediction
    print("R^2: %.2f" % r2_score(y_train, y_pred))
    y_pred = regr.predict(X_test)
    print("Test")
    print("MSE: %.2f" % mean_squared_error(y_test, y_pred))
    # MAE
    print("МАE: %.2f" % mean_absolute_error(y_test, y_pred))
    # RMSE
    print("RMSE: %.2f" % root_mean_squared_error(y_test, y_pred))
    # The coefficient of determination: 1 is perfect prediction
    print("R^2: %.2f" % r2_score(y_test, y_pred))

    regr_young =  RandomForestRegressor(n_estimators=300, max_features=15,  criterion='squared_error', random_state=random_state)
    weights = [1 if (item <= 50) else 0 for item in y_train]
    regr_young.fit(X_train, y_train, sample_weight=weights)

    regr_old =  RandomForestRegressor(n_estimators=300, max_features=15,  criterion='squared_error', random_state=random_state)
    weights = [1 if (item > 50) else 0 for item in y_train]
    regr_old.fit(X_train, y_train, sample_weight=weights)


    regr_old = OldRegressorCombined(n_estimators=300, max_features=15, criterion='squared_error', random_state=random_state)
    regr_young= YoungRegressorCombined(n_estimators=300, max_features=15, criterion='squared_error', random_state=random_state)
    regr =  RandomForestRegressor(n_estimators=300, max_features=15,  criterion='squared_error', random_state=random_state)

    stacking_regressor = StackingRegressor(estimators = [
        ("general", regr),
        ("young", regr_young),
        ("old", regr_old)
    ], final_estimator=RidgeCV())
    # Adding these weights actually fixes the angle for test, but decreases the score
    #stacking_regressor.fit(X_train, y_train, sample_weight=100* sample_weights*train_source_weights)
    stacking_regressor.fit(X_train, y_train)

    # Make predictions using the testing set
    #y_pred = regr.predict(X_train)
    #y_pred_young = regr.predict(X_train)
    #y_pred_old = regr.predict(X_train)
    y_pred = stacking_regressor.predict(X_train)
    #y_pred = (y_pred_young + y_pred_old) / 2
    #print("Coefficients: \n", np.round(regr.coef_, 2))

    regr = stacking_regressor.named_estimators_['general']
    # plot_model_estimation(regr, y_train, y_pred, X_train, feature_names=X_train.columns[list(reversed(np.argsort(regr.feature_importances_)))],  title=f"Data {cur_dataset} train", additional_feature=y_train_dataset)

    #y_pred = regr.predict(X_test)
    #y_pred_young = regr.predict(X_test)
    #y_pred_old = regr.predict(X_test)
    y_pred = stacking_regressor.predict(X_test)
    #y_pred = (y_pred_young + y_pred_old) / 2
    y_test_dataset = saved_data.loc[X_test.index].Dataset.values

    #print("Coefficients: \n", np.round(regr.coef_, 2))
    # plot_model_estimation(regr, y_test, y_pred, X_test, feature_names=X_train.columns[list(reversed(np.argsort(regr.feature_importances_)))], title=f"Data {cur_dataset} test", additional_feature=y_test_dataset)
    data_df = pd.DataFrame({"y_test": y_test, "y_pred": y_pred, "dataset": y_test_dataset})
   # sns.lmplot(data=data_df, x="y_test", y="y_pred",  hue="dataset", scatter=True,  ci=None, robust=True)

    coef_dict = {column: np.round(importance,2) for column, importance in reversed(sorted(zip(X_train.columns,regr.feature_importances_), key=lambda x:x[1]))}
    coef_df = pd.DataFrame({"cluster":coef_dict.keys(), "importance": coef_dict.values()}).transpose()
    regr = stacking_regressor

    print("Result model")
    print("Train")
    y_pred = regr.predict(X_train)
    print("MSE: %.2f" % mean_squared_error(y_train, y_pred))
    # MAE
    print("МАE: %.2f" % mean_absolute_error(y_train, y_pred))
    # RMSE
    print("RMSE: %.2f" % root_mean_squared_error(y_train, y_pred))
    # The coefficient of determination: 1 is perfect prediction
    print("R^2: %.2f" % r2_score(y_train, y_pred))
    y_pred = regr.predict(X_test)
    print("Test")
    print("MSE: %.2f" % mean_squared_error(y_test, y_pred))
    # MAE
    print("МАE: %.2f" % mean_absolute_error(y_test, y_pred))
    # RMSE
    print("RMSE: %.2f" % root_mean_squared_error(y_test, y_pred))
    # The coefficient of determination: 1 is perfect prediction
    print("R^2: %.2f" % r2_score(y_test, y_pred))

    result = permutation_importance(
        regr, X_test, y_test, n_repeats=10, random_state=random_state, n_jobs=2
    )
    # print(f"Elapsed time to compute the importances: {elapsed_time:.3f} seconds")
    forest_importances_test = pd.Series(result.importances_mean, index=feature_names)
    forest_impotances_test_sd = pd.Series(forest_importances_test.std(), index=feature_names)

    forest_importances_test.sort_values(ascending=False, inplace=True)
    forest_impotances_test_sd = forest_impotances_test_sd.loc[forest_importances_test.index]
    forest_importances_test = dict(forest_importances_test)
    forest_importances_test_sd = dict(forest_impotances_test_sd)


    result = permutation_importance(
        regr, X_test, y_test, n_repeats=10, random_state=random_state, n_jobs=2, scoring='neg_mean_absolute_error'
    )
    # print(f"Elapsed time to compute the importances: {elapsed_time:.3f} seconds")
    forest_importances_test_mae = pd.Series(result.importances_mean, index=feature_names)
    forest_impotances_test_mae_sd = pd.Series(forest_importances_test_mae.std(), index=feature_names)

    forest_importances_test_mae.sort_values(ascending=False, inplace=True)
    forest_impotances_test_mae_sd = forest_impotances_test_mae_sd.loc[forest_importances_test_mae.index]

    forest_importances_test_mae = dict(forest_importances_test_mae)
    forest_impotances_test_mae_sd = dict(forest_impotances_test_mae_sd)

    # result = permutation_importance(
    #     regr, X_train, y_train, n_repeats=10, random_state=random_state, n_jobs=2
    # )
    # print(f"Elapsed time to compute the importances: {elapsed_time:.3f} seconds")
    forest_importances_train = None
    forest_importances_train_sd = None
    #forest_importances_train = pd.Series(result.importances_mean, index=feature_names)
    #forest_importances_train_sd = pd.Series(forest_importances_train.std(), index=feature_names)
    #forest_importances_train.sort_values(ascending=False, inplace=True)
    #forest_importances_train_sd = forest_importances_train_sd.loc[forest_importances_train.index]
    #forest_importances_train = dict(forest_importances_train)
    #forest_importances_train_sd = dict(forest_importances_train_sd)
    return(
        forest_importances_test,
        forest_importances_test_mae,
        forest_importances_train,
        coef_dict,
        X_train,
        y_train,
        X_test,
        y_test,
        stacking_regressor,
        forest_importances_test_sd,
        forest_impotances_test_mae_sd,
        forest_importances_train_sd,
    )



In [ ]:
cur_dataset = "Combined"
result =   combined_regression_run(df, features_to_remove)
per_dataset_importances_test[cur_dataset] = result[0]
per_dataset_importances_mae[cur_dataset] = result[1]
#per_dataset_importances_train[cur_dataset] = result[2]
#per_dataset_coefs[cur_dataset] = result[3]
X_train = result[4]
y_train = result[5]
X_test = result[6]
y_test = result[7]
regr = result[8]
#per_dataset_importances_test_sd[cur_dataset]  = result[9]
#per_dataset_importances_sd_mae[cur_dataset]  = result[10]
#per_dataset_importances_sd_train[cur_dataset]  = result[11]

## Make final plots

In [ ]:
X_test = result[6]
y_test = result[7]
y_pred = regr.predict(X_test)
#y_pred = (y_pred_young + y_pred_old) / 2
y_test_dataset = df.loc[X_test.index].Dataset.values
plot_model_estimation(regr, y_test, y_pred, X_test,
                      feature_names=list(per_dataset_importances_test[cur_dataset].keys()),
                      title=f"Data {cur_dataset} test",
                      additional_feature=y_test_dataset,
                      point_size=30)
# jitter_amount = 1 # Adjust as needed
# y_pred = y_pred + np.random.uniform(-jitter_amount, jitter_amount, len(y_pred))
# y_test = y_test + np.random.uniform(-jitter_amount, jitter_amount, len(y_test))

data_df = pd.DataFrame({"y_test": y_test, "y_pred": y_pred, "dataset": y_test_dataset})

sns.lmplot(data=data_df, x="y_test", y="y_pred",  hue="dataset", scatter=True,  ci=None, robust=True)

# Plot importance table

In [ ]:
def plot_importances(per_dataset_importances, title="Importance table"):
    separate_coefficients = pd.DataFrame(per_dataset_importances)
    separate_coefficients['row_mean'] = separate_coefficients.mean(axis=1).values
    sorted_df = separate_coefficients.sort_values(by=['row_mean'], ascending=False)
    fig, ax = plt.subplots(figsize=(10, 12))
    sns.heatmap(sorted_df, annot=True, cmap="YlGnBu", ax=ax)
    fig.suptitle(title)


In [ ]:
plot_importances(per_dataset_importances_test, "RMSE Test importances")
#plot_importances(per_dataset_importances_train, "RMSE Train importances")
#plot_importances(per_dataset_coefs, "Single regression importances")
plot_importances(per_dataset_importances_mae, "MAE Test importances")

## Save results to files

In [ ]:
X_train = result[4]
y_train = result[5]
X_test = result[6]
y_test = result[7]
regr = result[8]

X_train = result[4]
y_train = result[5]
X_test = result[6]
y_test = result[7]
regr = result[8]
y_pred = regr.predict(X_test)

data_test = df.loc[X_test.index]
data_test['predicted_age'] = y_pred
data_test.to_csv("test_predictions_all_features.csv")
y_pred = regr.predict(X_train)

data_train = df.loc[X_train.index]
data_train['predicted_age'] = y_pred
data_train.to_csv("train_predictions_all_features.csv")


mae_importances_df = pd.DataFrame(per_dataset_importances_mae)
mae_importances_df.to_csv("importances_result_all_features.csv")



# Try the same with selected features

In [ ]:
one_hot_encoded_features = pd.get_dummies(df[['Sex', 'Ethnicity']], columns=['Sex', 'Ethnicity'], drop_first=True)

selected_by_marina = {'CD4 RTE', 'CD4 Th2', 'CD4 HLA-DR+ memory', ' CD4 Treg memory', 'CD8 RTE', 'CD8 Naive',
                      'CD8 Tcm CCR4+', 'CD8 Tem GZMK+', 'CD8 Tem GZMB+', 'CD8 HLA-DR+', 'CD8 Tmem KLRC2+',
                      'gd Vd1 GZMB+', 'NK CD56bright'}
selected_by_marina = {'CD4 RTE', 'CD4 Th2', 'CD4 HLA-DR+ memory', ' CD4 Treg memory', 'CD8 RTE', 'CD8 Naive', 'CD8 Tcm CCR4+', 'CD8 Tem GZMK+', 'CD8 Tem GZMB+', 'CD8 HLA-DR+', 'CD8 Tmem KLRC2+', "gd gd naive" , "gd Vd1 GZMK+" , 'gd Vd1 GZMB+', "gd Vd2 GZMK+" , 'NK CD56bright', "Non-classical monocytes", "CD8 Temra" , "CD4 Temra", "CD4 Terminal effector"}


features_to_remove =   [item for item in df.columns if item not in selected_by_marina ] + list(one_hot_encoded_features.columns)  + ['donor_id', 'Dataset', 'Sex', 'Ethnicity']
features_to_remove  = [item for item in features_to_remove if item != 'Age' ] # keep Age so far
df[one_hot_encoded_features.columns] = one_hot_encoded_features.values


## Again separate

In [ ]:
result = train_separate_datasets_regressions(df, features_to_remove)
per_dataset_importances = result[0]
per_dataset_importances_mae = result[1]
#per_dataset_importances_train = result[2]
#per_dataset_coefs = result[3]


## And combined

In [ ]:
cur_dataset = "Combined"
result =   combined_regression_run(df, features_to_remove)
per_dataset_importances[cur_dataset] = result[0]
per_dataset_importances_mae[cur_dataset] = result[1]
#per_dataset_importances_train[cur_dataset] = result[2]
#per_dataset_coefs[cur_dataset] = result[3]
X_train = result[4]
y_train = result[5]
X_test = result[6]
y_test = result[7]
regr = result[8]

In [ ]:
X_train.columns

## Make final plots

In [ ]:
X_test = result[6]
y_test = result[7]
y_pred = regr.predict(X_test)
#y_pred = (y_pred_young + y_pred_old) / 2
y_test_dataset = df.loc[X_test.index].Dataset.values
jitter_amount = 1 # Adjust as needed
y_pred = y_pred + np.random.uniform(-jitter_amount, jitter_amount, len(y_pred))
y_test = y_test + np.random.uniform(-jitter_amount, jitter_amount, len(y_test))

plot_model_estimation(regr, y_test, y_pred, X_test,
                      feature_names=list(per_dataset_importances[cur_dataset].keys()),
                      title=f"Data {cur_dataset} test",
                      additional_feature=y_test_dataset,
                      point_size=10)

data_df = pd.DataFrame({"y_test": y_test, "y_pred": y_pred, "dataset": y_test_dataset})

sns.lmplot(data=data_df, x="y_test", y="y_pred",  hue="dataset", scatter=True,  ci=None, robust=True)

# Plot importance table

In [ ]:
def plot_importances(per_dataset_importances, title="Importance table"):
    separate_coefficients = pd.DataFrame(per_dataset_importances)
    separate_coefficients['row_mean'] = separate_coefficients.mean(axis=1).values
    sorted_df = separate_coefficients.sort_values(by=['row_mean'], ascending=False)
    fig, ax = plt.subplots(figsize=(10, 12))
    sns.heatmap(sorted_df, annot=True, cmap="YlGnBu", ax=ax)
    fig.suptitle(title)


In [ ]:
plot_importances(per_dataset_importances, "RMSE Test importances")
#plot_importances(per_dataset_importances_train, "RMSE Train importances")
#plot_importances(per_dataset_coefs, "Single regression importances")
plot_importances(per_dataset_importances_mae, "MAE Test importances")

## Save table

In [ ]:

X_train = result[4]
y_train = result[5]
X_test = result[6]
y_test = result[7]
regr = result[8]
y_pred = regr.predict(X_test)

data_test = df.loc[X_test.index]
data_test['predicted_age'] = y_pred
data_test.to_csv("test_predictions_reduced_features.csv")
y_pred = regr.predict(X_train)

data_train = df.loc[X_train.index]
data_train['predicted_age'] = y_pred
data_train.to_csv("train_predictions_reduced_features.csv")


mae_importances_df = pd.DataFrame(per_dataset_importances_mae)
mae_importances_df.to_csv("importances_result_reduced_features.csv")



In [ ]:
X_train.columns

# Add sex/ ethnicity


In [ ]:
# features_to_remove = ['donor_id', 'Dataset', 'Sex', 'Ethnicity']
#
# one_hot_encoded_features = pd.get_dummies(df[['Sex', 'Ethnicity']], columns=['Sex', 'Ethnicity'], drop_first=True)
# df[one_hot_encoded_features.columns] = one_hot_encoded_features.values

In [ ]:
# df[one_hot_encoded_features.columns]

## Again separate

In [ ]:
# result = train_separate_datasets_regressions(df, features_to_remove)
# per_dataset_importances = result[0]
# per_dataset_importances_mae = result[1]
# per_dataset_importances_train = result[2]
# per_dataset_coefs = result[3]
#

## And combined

In [ ]:
# cur_dataset = "Combined"
# result =   combined_regression_run(df, features_to_remove)
# per_dataset_importances[cur_dataset] = result[0]
# per_dataset_importances_mae[cur_dataset] = result[1]
# per_dataset_importances_train[cur_dataset] = result[2]
# per_dataset_coefs[cur_dataset] = result[3]
# X_train = result[4]
# y_train = result[5]
# X_test = result[6]
# y_test = result[7]
# regr = result[8]

## Make final plots

In [ ]:
# X_test = result[6]
# y_test = result[7]
# y_pred = regr.predict(X_test)
# #y_pred = (y_pred_young + y_pred_old) / 2
# y_test_dataset = df.loc[X_test.index].Dataset.values
# plot_model_estimation(regr, y_test, y_pred, X_test,
#                       feature_names=list(per_dataset_importances[cur_dataset].keys()),
#                       title=f"Data {cur_dataset} test",
#                       additional_feature=y_test_dataset,
#                       point_size=30)
# # jitter_amount = 1 # Adjust as needed
# # y_pred = y_pred + np.random.uniform(-jitter_amount, jitter_amount, len(y_pred))
# # y_test = y_test + np.random.uniform(-jitter_amount, jitter_amount, len(y_test))
#
# data_df = pd.DataFrame({"y_test": y_test, "y_pred": y_pred, "dataset": y_test_dataset})
#
# sns.lmplot(data=data_df, x="y_test", y="y_pred",  hue="dataset", scatter=True,  ci=None, robust=True)

## Plot importance table

In [ ]:
# def plot_importances(per_dataset_importances, title="Importance table"):
#     separate_coefficients = pd.DataFrame(per_dataset_importances)
#     #separate_coefficients['row_mean'] = separate_coefficients.mean(axis=1).values
#     sorted_df = separate_coefficients.sort_values(by=['Combined'], ascending=False)
#     fig, ax = plt.subplots(figsize=(10, 12))
#     sns.heatmap(sorted_df, annot=True, cmap="YlGnBu", ax=ax)
#     fig.suptitle(title)
#

In [ ]:
# plot_importances(per_dataset_importances, "RMSE Test importances")
# plot_importances(per_dataset_importances_train, "RMSE Train importances")
# plot_importances(per_dataset_coefs, "Single regression importances")
# plot_importances(per_dataset_importances_mae, "MAE Test importances")

# Try the same with selected features + Sexx Ethnicity

In [ ]:
# one_hot_encoded_features = pd.get_dummies(df[['Sex', 'Ethnicity']], columns=['Sex', 'Ethnicity'], drop_first=True)
# selected_by_marina = set(['CD4 RTE', 'CD4 Th2', 'CD4 HLA-DR+ memory', ' CD4 Treg memory', 'CD8 RTE', 'CD8 Naive', 'CD8 Tcm CCR4+', 'CD8 Tem GZMK+', 'CD8 Tem GZMB+', 'CD8 HLA-DR+', 'CD8 Tmem KLRC2+', 'gd Vd1 GZMB+', 'NK CD56bright'] + list(one_hot_encoded_features.columns.values))
#
#
# features_to_remove =   [item for item in df.columns if item not in selected_by_marina ]  + ['donor_id', 'Dataset', 'Sex', 'Ethnicity']
# features_to_remove  = [item for item in features_to_remove if item != 'Age' ] # keep Age so far
# df[one_hot_encoded_features.columns] = one_hot_encoded_features.values
#

## Again separate

In [ ]:
# result = train_separate_datasets_regressions(df, features_to_remove)
# per_dataset_importances = result[0]
# per_dataset_importances_mae = result[1]
# per_dataset_importances_train = result[2]
# per_dataset_coefs = result[3]
#

## And combined

In [ ]:
# cur_dataset = "Combined"
# result =   combined_regression_run(df, features_to_remove)
# per_dataset_importances[cur_dataset] = result[0]
# per_dataset_importances_mae[cur_dataset] = result[1]
# per_dataset_importances_train[cur_dataset] = result[2]
# per_dataset_coefs[cur_dataset] = result[3]
# X_train = result[4]
# y_train = result[5]
# X_test = result[6]
# y_test = result[7]
# regr = result[8]

## Make final plots

In [ ]:
# X_test = result[6]
# y_test = result[7]
# y_pred = regr.predict(X_test)
# #y_pred = (y_pred_young + y_pred_old) / 2
# y_test_dataset = df.loc[X_test.index].Dataset.values
# jitter_amount = 1 # Adjust as needed
# y_pred = y_pred + np.random.uniform(-jitter_amount, jitter_amount, len(y_pred))
# y_test = y_test + np.random.uniform(-jitter_amount, jitter_amount, len(y_test))
#
# plot_model_estimation(regr, y_test, y_pred, X_test,
#                       feature_names=list(per_dataset_importances[cur_dataset].keys()),
#                       title=f"Data {cur_dataset} test",
#                       additional_feature=y_test_dataset,
#                       point_size=10)
#
# data_df = pd.DataFrame({"y_test": y_test, "y_pred": y_pred, "dataset": y_test_dataset})
#
# sns.lmplot(data=data_df, x="y_test", y="y_pred",  hue="dataset", scatter=True,  ci=None, robust=True)

## Plot importance table

In [ ]:
# def plot_importances(per_dataset_importances, title="Importance table"):
#     separate_coefficients = pd.DataFrame(per_dataset_importances)
#     separate_coefficients['row_mean'] = separate_coefficients.mean(axis=1).values
#     sorted_df = separate_coefficients.sort_values(by=['row_mean'], ascending=False)
#     fig, ax = plt.subplots(figsize=(10, 12))
#     sns.heatmap(sorted_df, annot=True, cmap="YlGnBu", ax=ax)
#     fig.suptitle(title)
#

In [ ]:
# plot_importances(per_dataset_importances, "RMSE Test importances")
# plot_importances(per_dataset_importances_train, "RMSE Train importances")
# plot_importances(per_dataset_coefs, "Single regression importances")
# plot_importances(per_dataset_importances_mae, "MAE Test importances")

## Save table

In [ ]:
# y_pred = regr.predict(X_test)
# X_train = result[4]
# y_train = result[5]
# X_test = result[6]
# y_test = result[7]
# regr = result[8]

In [ ]:
# selected_data = df.loc[X_test.index]

In [ ]:
# selected_data['predicted_age'] = y_pred

In [ ]:
# selected_data.to_csv("test_predictions.csv")

In [ ]:
# selected_data